# MTLnet — Colab Training

Multi-task learning for POI prediction (category + next-POI).

**Typical workflow:**
1. Run **Setup** (sections 1–5) once per session
2. Set your **Pipeline Configuration** (section 6)
3. Run **Step 1** → Generate Embeddings
4. Run **Step 2** → Create Model Inputs
5. Run **Step 3** → Train

**Drive layout expected:**
```
MyDrive/mestrado/PoiMtlNet/
  data/checkins/      ← raw checkins (Florida.parquet, Alabama.parquet, …)
  data/miscellaneous/ ← shapefiles (tl_2022_*_tract_*/)
  output/             ← generated embeddings and model inputs  [auto-created]
  results/            ← training results                        [auto-created]
```

---
## ① Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive/mestrado/PoiMtlNet')
DATA_ROOT    = DRIVE_ROOT / 'data'
OUTPUT_DIR   = DRIVE_ROOT / 'output'
RESULTS_ROOT = DRIVE_ROOT / 'results'

for p in [DATA_ROOT, OUTPUT_DIR, RESULTS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['DATA_ROOT']    = str(DATA_ROOT)
os.environ['OUTPUT_DIR']   = str(OUTPUT_DIR)
os.environ['RESULTS_ROOT'] = str(RESULTS_ROOT)

print(f'DATA_ROOT:    {DATA_ROOT}  (exists={DATA_ROOT.exists()})')
print(f'OUTPUT_DIR:   {OUTPUT_DIR}  (exists={OUTPUT_DIR.exists()})')
print(f'RESULTS_ROOT: {RESULTS_ROOT}  (exists={RESULTS_ROOT.exists()})')

DATA_ROOT:    /content/drive/MyDrive/mestrado/PoiMtlNet/data  (exists=True)
OUTPUT_DIR:   /content/drive/MyDrive/mestrado/PoiMtlNet/output  (exists=True)
RESULTS_ROOT: /content/drive/MyDrive/mestrado/PoiMtlNet/results  (exists=True)


In [5]:
REPO_DIR = Path('/content/PoiMtlNet')

if not REPO_DIR.exists():
    !git clone https://github.com/VitorHugoOli/PoiMtlNet.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git log --oneline -3

remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 58 (delta 45), reused 54 (delta 45), pack-reused 0 (from 0)
Unpacking objects: 100% (58/58), 22.03 KiB | 1.00 MiB/s, done.
From https://github.com/VitorHugoOli/PoiMtlNet
   1c72d5d..773c393  main       -> origin/main
   d898a95..a5c9bbe  copilot/create-improvement-plan-for-mtlnet -> origin/copilot/create-improvement-plan-for-mtlnet
Updating 1c72d5d..773c393
Fast-forward
 notebooks/colab_training.ipynb | 143 ++++++++++++++++++++++++++++++++++++-----
 1 file changed, 128 insertions(+), 15 deletions(-)
/content/PoiMtlNet
773c393 (HEAD -> main, origin/main, origin/HEAD) Criado usando o Colab
1c72d5d feat: some improvements
6aa7b1c Created using Colab


In [6]:
# Base requirements
!pip install -q -r requirements_colab.txt

# NashMTL requires cvxpy + ECOS solver
!pip install -q cvxpy

# PyTorch Geometric — needed for HGI, DGI, POI2HGI, Check2HGI
!pip install -q torch-geometric

!pip install -q pytorch_warmup

import torch

!pip uninstall torch-scatter torch-sparse torch-geometric torch-cluster  --y
!pip install torch-scatter -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-cluster -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install git+https://github.com/pyg-team/pytorch_geometric.git

Found existing installation: torch-geometric 2.7.0
Uninstalling torch-geometric-2.7.0:
  Successfully uninstalled torch-geometric-2.7.0
Looking in links: https://data.pyg.org/whl/torch-2.6.0+cu124.html
  Using cached https://data.pyg.org/whl/torch-2.6.0%2Bcu124/torch_scatter-2.1.2%2Bpt26cu124-cp312-cp312-linux_x86_64.whl (10.8 MB)
Looking in links: https://data.pyg.org/whl/torch-2.6.0+cu124.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 3.7 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.6.0+cu124.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 89.4 MB/s eta 0:00:00
  Cloning https://github.com/pyg-team/pytorch_geometric.git to /tmp/pip-req-build-74layry2
  Running command git clone --filter=blob:none --quiet https://github.com/pyg-team/pytorch_geometric.git /tmp/pip-req-build-74layry2
  Resolved https://github.com/pyg-team/pytorch_geometric.git to commit c9c18c97a05af228281471027d44cfd328afebba
  Installing build dependencies ... done
 

In [7]:
import sys

for sub in ('src', 'research'):
    p = str(REPO_DIR / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

from configs.globals import DEVICE
print('Device:', DEVICE)

Device: cuda


---
## ② Pipeline Configuration

**Edit this cell before running any pipeline step.**  
All subsequent cells read `STATE`, `ENGINE`, and `TASK` from here.

In [8]:
# ── edit here ─────────────────────────────────────────────────────────────────
STATE  = 'texas'   # florida | alabama | georgia | california | texas | arizona
ENGINE = 'hgi'       # hgi | dgi | poi2hgi | check2hgi | time2vec | space2vec | sphere2vec | fusion
TASK   = 'mtl'       # mtl | category | next

# Optional training overrides (None = use config defaults)
EPOCHS = None        # e.g. 50
FOLDS  = None        # e.g. 1 for a quick smoke test
# ──────────────────────────────────────────────────────────────────────────────

from configs.paths import EmbeddingEngine, IoPaths, Resources
ENGINE_ENUM = EmbeddingEngine(ENGINE)

# Map state name to shapefile resource (needed by graph-based engines)
_SHAPEFILES = {
    'florida':    Resources.TL_FL,
    'alabama':    Resources.TL_AL,
    'georgia':    Resources.TL_GA,
    'california': Resources.TL_CA,
    'texas':      Resources.TL_TX,
    'arizona':    Resources.TL_AZ,
}
SHAPEFILE = _SHAPEFILES.get(STATE.lower())

print(f'State:     {STATE}')
print(f'Engine:    {ENGINE}')
print(f'Task:      {TASK}')
print(f'Shapefile: {SHAPEFILE}')
print(f'Epochs:    {EPOCHS or "(default)"}')
print(f'Folds:     {FOLDS  or "(all)"}')

State:     texas
Engine:    hgi
Task:      mtl
Shapefile: /content/drive/MyDrive/mestrado/PoiMtlNet/data/miscellaneous/tl_2022_48_tract_TX/tl_2022_48_tract.shp
Epochs:    (default)
Folds:     (all)


---
## ③ Step 1 — Generate Embeddings

Run the subsection that matches your `ENGINE`.  
Skip this step entirely if embeddings are already in `output/` on Drive.

### HGI / POI2HGI / Check2HGI  (`engine = hgi | poi2hgi | check2hgi`)

In [ ]:
# Run only if ENGINE in ('hgi', 'poi2hgi', 'check2hgi')
assert ENGINE in ('hgi', 'poi2hgi', 'check2hgi'), f'Wrong engine: {ENGINE}'
assert SHAPEFILE is not None, f'No shapefile registered for state "{STATE}"'

import importlib.util
from copy import copy

# Load the correct pipeline module without executing __main__
_pipe_path = REPO_DIR / f'pipelines/embedding/{ENGINE}.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

# Patch STATES to run only our chosen state
_pipe.STATES = {
    STATE.capitalize(): {'shapefile': SHAPEFILE},
}

_pipe.run_pipeline()

  Cross-region edge weight w_r = 0.7
Reading POI data...
Reading boroughs data...
  Encoded 7 categories: ['Community', 'Entertainment', 'Food', 'Nightlife', 'Outdoors']...
  Encoded 365 fclasses: ['24 Hour Fitness', 'AMC Theatre', 'AT&T', 'Accessories', 'Administration']...
Creating spatial graph...
Computing region features...
Loading POI embeddings...
  Saved encodings to /content/drive/MyDrive/mestrado/PoiMtlNet/output/hgi/texas/temp/encodings.json
✓ Phase 3a complete: Delaunay graph created
  POIs: 160938, Regions: 6553, Edges: 482789
  Intermediate files saved to: /content/drive/MyDrive/mestrado/PoiMtlNet/output/hgi/texas/temp
    - edges.csv (Delaunay graph)
    - pois.csv (POI-region mapping)
    - encodings.json (category/fclass mappings)
POI2Vec Training: Texas
Loading edges and POIs...
Initializing Node2Vec walk generator...
  POIs: 160938
  Edges: 482789
  Unique fclass values: 365
PHASE 3b: Generating Random Walks
Generating walks (batch_size=128)...


Walk generation: 100%|██████████| 1258/1258 [07:28<00:00,  2.81it/s]


Generated 4828140 fclass walks
Building global co-occurrence statistics...
Co-occurrence stats computed for 365 fclass values

PHASE 3c: Training fclass Embeddings
Extracting category-fclass hierarchy pairs...
  Found 365 unique (category, fclass) pairs
Building POISet dataset...
Initializing EmbeddingModel...
Training for 100 epochs (batch=2048, lr=0.05, workers=10, device=cuda:None)...


Epoch 1/100: 100%|██████████| 2358/2358 [00:34<00:00, 67.38it/s, loss=0.0533, h_loss=5.49e-04]


  Epoch 1/100: Loss=0.1526, Hierarchy Loss=3.07e-04


...

Epoch 29/100: 100%|██████████| 2358/2358 [00:31<00:00, 75.82it/s, loss=0.1744, h_loss=9.81e-03]


  Epoch 29/100: Loss=0.0464, Hierarchy Loss=9.72e-03


Epoch 30/100: 100%|██████████| 2358/2358 [00:29<00:00, 80.76it/s, loss=0.0102, h_loss=1.02e-02]


  Epoch 30/100: Loss=0.0449, Hierarchy Loss=9.98e-03


Epoch 31/100:  47%|████▋     | 1118/2358 [00:12<00:13, 93.73it/s, loss=0.0103, h_loss=1.03e-02]

### DGI  (`engine = dgi`)

In [ ]:
assert ENGINE == 'dgi', f'Wrong engine: {ENGINE}'
assert SHAPEFILE is not None, f'No shapefile registered for state "{STATE}"'

import importlib.util

_pipe_path = REPO_DIR / 'pipelines/embedding/dgi.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

_pipe.STATES = {
    STATE.capitalize(): {'shapefile': SHAPEFILE},
}

_pipe.run_pipeline()

### Time2Vec  (`engine = time2vec`)

In [ ]:
assert ENGINE == 'time2vec', f'Wrong engine: {ENGINE}'

import importlib.util

_pipe_path = REPO_DIR / 'pipelines/embedding/time2vec.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

_pipe.STATES = {STATE.capitalize(): {}}

_pipe.run_pipeline()

### Space2Vec / Sphere2Vec  (`engine = space2vec | sphere2vec`)

In [ ]:
assert ENGINE in ('space2vec', 'sphere2vec'), f'Wrong engine: {ENGINE}'

import importlib.util

_pipe_path = REPO_DIR / f'pipelines/embedding/{ENGINE}.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

_pipe.STATES = {STATE.capitalize(): {}}

_pipe.run_pipeline()

### Fusion  (`engine = fusion`)

Fusion concatenates multiple pre-existing embeddings — run after each component engine is done.

In [ ]:
assert ENGINE == 'fusion', f'Wrong engine: {ENGINE}'

import importlib.util

_pipe_path = REPO_DIR / 'pipelines/fusion.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

_pipe.STATES = {STATE.capitalize(): {}}

_pipe.run_pipeline()

---
## ④ Step 2 — Create Model Inputs

Generates `category.parquet` and `next.parquet` inside `output/{engine}/{state}/input/`.
Skip if inputs are already present.

In [ ]:
import importlib.util

_pipe_path = REPO_DIR / 'pipelines/create_inputs.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

_pipe.STATES = {
    STATE.capitalize(): {
        'engine': ENGINE_ENUM,
        'use_checkin': ENGINE == 'time2vec',  # time2vec uses check-in-level embeddings
    }
}

_pipe.run_pipeline()

In [ ]:
# Verify inputs exist before training
cat_path  = IoPaths.get_category(STATE, ENGINE_ENUM)
next_path = IoPaths.get_next(STATE, ENGINE_ENUM)

print(f'Category input  ({"OK" if cat_path.exists() else "MISSING"}): {cat_path}')
print(f'Next-POI input  ({"OK" if next_path.exists() else "MISSING"}): {next_path}')

---
## ⑤ Step 3 — Train

Results (metrics CSVs, classification reports, plots, checkpoints) are written to Drive under `results/`.

In [ ]:
# Build the training command from the config set in section ②
_cmd = f'python scripts/train.py --task {TASK} --state {STATE} --engine {ENGINE}'
if EPOCHS is not None:
    _cmd += f' --epochs {EPOCHS}'
if FOLDS is not None:
    _cmd += f' --folds {FOLDS}'

print('Command:', _cmd)

In [ ]:
# Full run
!{_cmd}

In [ ]:
# Smoke test — 1 fold, 5 epochs (confirm setup is correct before the full run)
!python scripts/train.py --task {TASK} --state {STATE} --engine {ENGINE} --folds 1 --epochs 5

---
## ⑥ Inspect Results

In [ ]:
import pandas as pd

results_dir = IoPaths.get_results_dir(STATE, ENGINE_ENUM)
print('Results directory:', results_dir)
print()

csv_files = sorted(results_dir.glob('*.csv'))
if csv_files:
    for f in csv_files:
        print(f'--- {f.name} ---')
        print(pd.read_csv(f).to_string(index=False))
        print()
else:
    print('No CSV results found yet.')